# 08 - Publication Figures & Visual Storyboard Pipeline

This notebook generates the **Publication Figures** and **Thesis Visual Storyboard** across all evaluated dimensions (, , ).

### Global Data Integrity & Design Rules
1. **Explicit Model Isolation**:  variants (, , , ) are strictly separated from  and classical baselines (, , ).
2. **Representative Benchmark Suite**: Evaluations focus on the 5 canonical BBOB functions:  (Sphere),  (Rosenbrock),  (Discus),  (Rastrigin),  (Gallagher).
3. **Condition Isolation**: Clean (σ=0.0) and Noisy (σ=0.05) runs are evaluated independently without cross-condition pooling.
4. **Publication Standards**: High-DPI exports (300 DPI), consistent typography (Inter/Helvetica), clean margins, and distinct color palettes.

---
### Hierarchy & Pipeline Structure ()
- **** (Cross-Solver & Multi-Model Comparative Figures)
  - : (RQ1) Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
  - : (RQ3) Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
  - : Problem hardness category success rates & Landscape Fragility Index matrix.
- **** (LLaMEA-14B Model-Specific Figures)
  - : (RQ2/3 Ablation) Prompt Scaffolding on 14B.
  -  & : Convergence trajectories & empirical target hit rate ECDFs.
- **** (LLaMEA-7B Model-Specific Figures - reserved for 5D completion)

In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, RESULTS_DIR
from infra.storage import get_db_connection

IOH_LOGS_DIR = DATA_DIR / 'ioh_logs'
DB_PATH      = DATA_DIR / 'db.sqlite3'
FIGURES_DIR  = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Set global plotting aesthetic
plt.rcParams['font.sans-serif'] = ['Inter', 'Helvetica', 'DejaVu Sans', 'Arial']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

# Dynamically query raw models from DB
with get_db_connection() as conn:
    _raw_models = pd.read_sql_query("SELECT DISTINCT llm_name FROM experiments WHERE status = 'completed'", conn)['llm_name'].dropna().tolist()

DISCOVERED_MODELS = sorted(list(set(_raw_models)))
print(f'🤖 Dynamically discovered models in DB: {DISCOVERED_MODELS}')

def comparative_dir(dim: int) -> Path:
    p = FIGURES_DIR / 'comparative' / f'{dim}D'
    p.mkdir(parents=True, exist_ok=True)
    return p

def model_fig_dir(llm_name: str, dim: int) -> Path:
    folder_name = llm_name.removesuffix('.gguf')
    p = FIGURES_DIR / folder_name / f'{dim}D'
    p.mkdir(parents=True, exist_ok=True)
    return p

def model_std_dir(llm_name: str, dim: int, noise_std: float) -> Path:
    p = model_fig_dir(llm_name, dim) / f'std_{noise_std}'
    p.mkdir(parents=True, exist_ok=True)
    return p

print('✅ Figure environment initialized (PNG-only export, 300 DPI).')


🤖 Dynamically discovered models in DB: ['qwen2.5-coder-14b-instruct-q4_k_m.gguf', 'qwen2.5-coder-7b-instruct-q4_k_m.gguf']
✅ Figure environment initialized (PNG-only export, 300 DPI).


In [2]:
PROBLEM_IDS = [1, 8, 11, 15, 21]

# ── 1. Global Constants & Aesthetic Dictionaries ────────────────────────
ALL_SOLVERS_ORDER = [
    "LLaMEA-14B / baseline",
    "LLaMEA-14B / guided",
    "LLaMEA-14B / thinking",
    "LLaMEA-14B / vectorization",
    "LLaMEA-7B / baseline",
    "CMA-ES",
    "DE",
    "PSO",
]

SOLVERS_14B = ["LLaMEA-14B / baseline", "LLaMEA-14B / guided", "LLaMEA-14B / thinking", "LLaMEA-14B / vectorization"]
SOLVERS_7B = ["LLaMEA-7B / baseline"]
SOLVERS_CLASSICAL = ["CMA-ES", "DE", "PSO"]

SOLVER_COLORS = {
    "LLaMEA-14B / baseline":      "#1f77b4",
    "LLaMEA-14B / guided":        "#ff7f0e",
    "LLaMEA-14B / thinking":      "#2ca02c",
    "LLaMEA-14B / vectorization": "#d62728",
    "LLaMEA-7B / baseline":       "#9467bd",
    "CMA-ES":                     "#8c564b",
    "DE":                         "#e377c2",
    "PSO":                        "#7f7f7f",
}

BBOB_NAMES_MAP = {
    1: "Sphere (f1)", 8: "Rosenbrock (f8)", 11: "Discus (f11)",
    15: "Rastrigin (f15)", 21: "Gallagher (f21)"
}
BBOB_NAMES = BBOB_NAMES_MAP
BBOB_CLASSES_MAP = {
    1: "Separable", 8: "Low Conditioning", 11: "High Conditioning",
    15: "Multi-Modal (Global)", 21: "Multi-Modal (Weak)"
}
BBOB_CLASSES = BBOB_CLASSES_MAP

# ── 2. Data Parsers for IOH Logs & SQLite Database ───────────────────────────
def parse_ioh_dat_file(dat_path: Path):
    runs = []
    current_evals, current_raw = [], []
    with open(dat_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith(("function", "evaluations", "#", "instance")):
                if current_evals: runs.append((np.array(current_evals), np.array(current_raw))); current_evals, current_raw = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                try: current_evals.append(float(parts[0])); current_raw.append(float(parts[1]))
                except ValueError: continue
    if current_evals: runs.append((np.array(current_evals), np.array(current_raw)))
    return runs

def resolve_solver_name(parent_name: str) -> str:
    p = parent_name.lower()
    if "cmaes" in p or "cma_es" in p: return "CMA-ES"
    if "pso" in p: return "PSO"
    if p == "de" or p.startswith(("de_", "de-")) or "_de_" in p: return "DE"
    if "7b" in p: return "LLaMEA-7B / baseline"
    if "thinking" in p: return "LLaMEA-14B / thinking"
    if "vectorization" in p: return "LLaMEA-14B / vectorization"
    if "guided" in p: return "LLaMEA-14B / guided"
    if "14b" in p or "baseline" in p or "llamea" in p: return "LLaMEA-14B / baseline"
    return parent_name

def load_benchmark_ioh_data(ioh_dir: Path):
    data_store = {}
    if not ioh_dir.exists(): return data_store
    for json_path in ioh_dir.glob("**/*.json"):
        try:
            with open(json_path, "r") as jf: meta = json.load(jf)
        except Exception: continue
        path_str = str(json_path.relative_to(ioh_dir))
        dim_m = re.search(r"(\d+)D", path_str); dim = int(dim_m.group(1)) if dim_m else None
        noise_m = re.search(r"std_([\d\.]+)", path_str); noise_std = float(noise_m.group(1)) if noise_m else 0.0
        p_id = meta.get("function_id")
        if p_id is None: p_m = re.search(r"f(\d+)", path_str); p_id = int(p_m.group(1)) if p_m else None
        parent_name = json_path.parent.name
        if "dummy" in parent_name.lower(): continue
        solver_name = resolve_solver_name(parent_name)
        for sc in meta.get("scenarios", []):
            if dim is None: dim = sc.get("dimension")
            if p_id is None or dim is None: continue
            key = (dim, noise_std, p_id)
            if key not in data_store: data_store[key] = {}
            if solver_name not in data_store[key]: data_store[key][solver_name] = []
            dat_p = sc.get("path")
            if dat_p and (json_path.parent / dat_p).exists(): data_store[key][solver_name].extend(parse_ioh_dat_file(json_path.parent / dat_p))
    return data_store

def load_sqlite_synthesis_data(db_path: Path):
    if not db_path.exists(): return pd.DataFrame(), pd.DataFrame()
    conn = sqlite3.connect(db_path)
    df_exp = pd.read_sql_query("SELECT * FROM experiments", conn)
    df_iter = pd.read_sql_query("SELECT i.id AS iteration_id, i.experiment_id, i.algorithm_name, i.raw_fitness, i.final_error, i.timed_out, i.converged, i.runtime_seconds, e.problem_id, e.dim, e.mode, e.llm_name, e.prompt_strategy FROM iterations i JOIN experiments e ON i.experiment_id = e.id", conn)
    conn.close()
    
    def map_exp_solver(row):
        llm = str(row.get("llm_name", "")).lower()
        strat = str(row.get("prompt_strategy", "")).lower()
        if "7b" in llm:
            return "LLaMEA-7B / baseline"
        elif "14b" in llm:
            return f"LLaMEA-14B / {strat}"
        return f"LLaMEA / {strat}"
        
    if not df_exp.empty:
        df_exp["solver_name"] = df_exp.apply(map_exp_solver, axis=1)
    if not df_iter.empty:
        df_iter["solver_name"] = df_iter.apply(map_exp_solver, axis=1)
        
    return df_exp, df_iter

print("✅ Parsers loaded successfully.")


✅ Parsers loaded successfully.


In [3]:
# ── 3. Load Datasets from SQLite and IOH Logs ─────────────────────────────────
df_exp, df_iter = load_sqlite_synthesis_data(DB_PATH)
all_benchmark_data = load_benchmark_ioh_data(IOH_LOGS_DIR)
all_solvers = [s for s in ALL_SOLVERS_ORDER if any(s in cond for cond in all_benchmark_data.values())]
all_dims = sorted(list(set(k[0] for k in all_benchmark_data.keys())))
all_noise_stds = sorted(list(set(k[1] for k in all_benchmark_data.keys())))

print(f"📦 Loaded {len(all_benchmark_data)} benchmark problem conditions.")
print(f"   • Dimensions detected: {all_dims}")
print(f"   • Noise levels detected: {all_noise_stds}")
print(f"   • Solvers evaluated: {all_solvers}")


📦 Loaded 30 benchmark problem conditions.
   • Dimensions detected: [2, 3, 5]
   • Noise levels detected: [0.0, 0.05]
   • Solvers evaluated: ['LLaMEA-14B / baseline', 'LLaMEA-14B / guided', 'LLaMEA-14B / thinking', 'LLaMEA-14B / vectorization', 'LLaMEA-7B / baseline', 'CMA-ES', 'DE']


# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
clean_std = 0.0
noisy_std = 0.05

for dim in all_dims:


In [4]:
# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
clean_std = 0.0
noisy_std = 0.05

for dim in all_dims:
    fig_e = make_subplots(
        rows=2, cols=2,
        specs=[[{}, {}], [{"colspan": 2}, None]],
        subplot_titles=(
            f"<b>(A1) Mean Success Rate — Clean (σ={clean_std}, {dim}D)</b>",
            f"<b>(A2) Mean Success Rate — Noisy (σ={noisy_std}, {dim}D)</b>",
            f"<b>(B) Landscape Fragility Index Matrix (Clean → Noisy σ={noisy_std} Degradation, {dim}D)</b>"
        ),
        vertical_spacing=0.18,
        horizontal_spacing=0.08,
        row_heights=[0.45, 0.55]
    )

    # Subplots A1 & A2 (Clean vs. Noisy Hardness Success Rates across Solvers)
    for c_idx, noise_std in enumerate([clean_std, noisy_std], start=1):
        prob_records = []
        for (d, n_std, p_id), solvers in all_benchmark_data.items():
            if d != dim or n_std != noise_std: continue
            h_class = BBOB_CLASSES_MAP.get(p_id, "Unknown")
            for solver_name, runs in solvers.items():
                if not runs: continue
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                if not finals: continue
                n_conv = sum(1 for f in finals if f < 1e-8)
                sr = n_conv / len(finals) * 100.0
                prob_records.append({"hardness_class": h_class, "solver_name": solver_name, "success_rate": sr})
        
        df_p = pd.DataFrame(prob_records)
        df_c = df_p.groupby(["hardness_class", "solver_name"])["success_rate"].mean().reset_index() if not df_p.empty else pd.DataFrame()
        
        for s in all_solvers:
            df_s = df_c[df_c["solver_name"] == s] if not df_c.empty else pd.DataFrame()
            class_order = ["Separable", "Low Conditioning", "High Conditioning", "Multi-Modal (Global)", "Multi-Modal (Weak)"]
            if not df_s.empty:
                df_s_map = dict(zip(df_s["hardness_class"], df_s["success_rate"]))
                y_vals = [df_s_map.get(c, 0.0) for c in class_order]
            else:
                y_vals = [0.0 for _ in class_order]
            
            fig_e.add_trace(
                go.Bar(
                    x=class_order, y=y_vals, name=s,
                    marker_color=SOLVER_COLORS.get(s, "#7f7f7f"),
                    showlegend=(c_idx == 1 and dim == all_dims[0])
                ),
                row=1, col=c_idx
            )
        
        fig_e.update_yaxes(title="<b>Success Rate (%)</b>" if c_idx == 1 else None, range=[0, 105], row=1, col=c_idx)
        fig_e.update_xaxes(title="<b>Landscape Hardness</b>", tickangle=-15, row=1, col=c_idx)
        fig_e.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False, row=1, col=c_idx)
        fig_e.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False, row=1, col=c_idx)
    
    # Subplot B: Fragility Heatmap
    fragility_rows = []
    prob_ids_order = [1, 8, 11, 15, 21]
    for s in all_solvers:
        s_row = []
        for p_id in prob_ids_order:
            runs_clean = all_benchmark_data.get((dim, clean_std, p_id), {}).get(s, [])
            runs_noisy = all_benchmark_data.get((dim, noisy_std, p_id), {}).get(s, [])
            err_c = np.median([r[1][-1] for r in runs_clean if len(r[1]) > 0]) if runs_clean else 1e-12
            err_n = np.median([r[1][-1] for r in runs_noisy if len(r[1]) > 0]) if runs_noisy else 1e-12
            log_c = np.log10(max(err_c, 1e-12))
            log_n = np.log10(max(err_n, 1e-12))
            s_row.append(round(float(log_n - log_c), 2))
        fragility_rows.append(s_row)
    
    p_labels = [f"{BBOB_NAMES_MAP.get(p, str(p))}<br>({BBOB_CLASSES_MAP.get(p, "")})" for p in prob_ids_order]
    fig_e.add_trace(
        go.Heatmap(
            z=fragility_rows, x=p_labels, y=all_solvers,
            colorscale="Blues", text=fragility_rows, texttemplate="%{text}",
            colorbar=dict(title="Δ log₁₀(Err)", len=0.45, y=0.22, yanchor="middle", x=1.02),
            showscale=True
        ),
        row=2, col=1
    )
    fig_e.update_xaxes(title="<b>BBOB Problem Suite (Hardness Ordered)</b>", row=2, col=1)
    fig_e.update_yaxes(title="<b>Evaluated Solver</b>", row=2, col=1)
    
    fig_e.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Problem Difficulty & Noise Sensitivity Dashboard (Clean vs. Noisy) — {dim}D</b>",
            x=0.02, y=0.985,
            font=dict(size=14, color="#2c3e50", family="Inter, Helvetica, Arial, sans-serif")
        ),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=11, color="#333333"),
        margin=dict(l=65, r=45, t=130, b=55),
        width=1150, height=920,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.04, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.92)", bordercolor="rgba(0,0,0,0.12)", borderwidth=1,
            font=dict(size=10.0)
        )
    )
    out_path = comparative_dir(dim) / "figure_e_difficulty_and_noise.png"
    fig_e.write_image(str(out_path), scale=3)

print("✅ Figure E generated in results/figures/comparative/{dim}D/.")


✅ Figure E generated in results/figures/comparative/{dim}D/.


### 📊 Model-Specific Hardness Success Rates (Clean vs. Noisy)
Separates the mean success rate analysis per LLM model (, ) across clean and noisy landscapes.

In [5]:
# ── Model-Specific Success Rate by Landscape Hardness (Clean vs. Noisy) ──
def render_model_success_rate_by_hardness(model_tag: str, solvers_list: list, dim: int):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'<b>(A) Clean Landscape (σ=0.0, {dim}D)</b>',
            f'<b>(B) Noisy Landscape (σ=0.05, {dim}D)</b>'
        ),
        horizontal_spacing=0.10
    )
    
    for c_idx, noise_std in enumerate([0.0, 0.05], start=1):
        prob_records = []
        for (d, n_std, p_id), solvers in all_benchmark_data.items():
            if d != dim or n_std != noise_std: continue
            h_class = BBOB_CLASSES_MAP.get(p_id, 'Unknown')
            for solver_name, runs in solvers.items():
                if solver_name not in solvers_list or not runs: continue
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                if not finals: continue
                n_conv = sum(1 for f in finals if f < 1e-8)
                sr = n_conv / len(finals) * 100.0
                prob_records.append({'hardness_class': h_class, 'solver_name': solver_name, 'success_rate': sr})
        
        df_p = pd.DataFrame(prob_records)
        df_c = df_p.groupby(['hardness_class', 'solver_name'])['success_rate'].mean().reset_index() if not df_p.empty else pd.DataFrame()
        
        class_order = ['Separable', 'Low Conditioning', 'High Conditioning', 'Multi-Modal (Global)', 'Multi-Modal (Weak)']
        for s in solvers_list:
            df_s = df_c[df_c['solver_name'] == s] if not df_c.empty else pd.DataFrame()
            if not df_s.empty:
                df_s_map = dict(zip(df_s['hardness_class'], df_s['success_rate']))
                y_vals = [df_s_map.get(c, 0.0) for c in class_order]
            else:
                y_vals = [0.0 for _ in class_order]
            
            clean_name = s.split('/')[-1].strip().capitalize()
            fig.add_trace(
                go.Bar(
                    x=class_order, y=y_vals, name=clean_name,
                    marker_color=SOLVER_COLORS.get(s, '#7f7f7f'),
                    showlegend=(c_idx == 1)
                ),
                row=1, col=c_idx
            )
        
        fig.update_yaxes(title='<b>Success Rate (%)</b>' if c_idx == 1 else None, range=[0, 105], row=1, col=c_idx)
        fig.update_xaxes(title='<b>Landscape Hardness Category</b>', tickangle=-20, row=1, col=c_idx)
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False, row=1, col=c_idx)
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False, row=1, col=c_idx)
    
    fig.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>LLaMEA ({model_tag}) Mean Success Rate by Landscape Hardness & Noise — {dim}D</b>',
            x=0.02, y=0.98,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#333333'),
        margin=dict(l=65, r=30, t=110, b=75),
        width=1100, height=480,
        legend=dict(
            orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5,
            bgcolor='rgba(255,255,255,0.92)', bordercolor='rgba(0,0,0,0.12)', borderwidth=1,
            font=dict(size=10.5)
        )
    )
    out_p = model_fig_dir(model_tag, dim) / 'figure_success_rate_by_hardness.png'
    fig.write_image(str(out_p), scale=3)

for model_name in DISCOVERED_MODELS:
    model_name = model_name.removesuffix('.gguf')
    solvers = [s for s in all_solvers if ('7b' in s.lower() if '7b' in model_name.lower() else '14b' in s.lower())]
    for dim in all_dims:
        render_model_success_rate_by_hardness(model_name, solvers, dim)

print('✅ Model-specific success rate by hardness generated for all DB models across all dimensions.')


✅ Model-specific success rate by hardness generated for all DB models across all dimensions.


# 🎓 Part II: Thesis Visual Storyboard (RQ1 → RQ2 → RQ3 → Scaffolding Narrative Chain)

The following four figures form the core visual evidence for the thesis, saved into `results/figures/{dim}D/thesis/`:
- **Figure 1 (RQ1):** Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2):** LLaMEA Synthesis Competency vs. Classical Baselines (Clean Convergence Trajectories & IQR).
- **Figure 3 (RQ3 Hero):** Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation):** Prompt Scaffolding Ablation on LLaMEA-14B (Baseline vs. Guided vs. Thinking vs. Vectorization).


In [6]:
# ── THESIS Figure 1: Benchmark Validation (RQ1: Stochastic Extension) ─────────
clean_std = 0.0
noisy_stds = [s for s in all_noise_stds if s > 0]
noisy_std = noisy_stds[0] if noisy_stds else 0.05

for dim in all_dims:
    clean_medians = []
    noisy_medians = []
    problem_labels = []
    
    for p_id in PROBLEM_IDS:
        p_name = BBOB_NAMES.get(p_id, f"f{p_id}")
        p_class = BBOB_CLASSES.get(p_id, "")
        problem_labels.append(f"<b>{p_name}</b><br><sup>{p_class}</sup>")
        
        # Clean terminal errors across all solvers
        c_key = (dim, clean_std, p_id)
        c_finals = []
        if c_key in all_benchmark_data:
            for s, runs in all_benchmark_data[c_key].items():
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        c_finals.append(raw_vals[-1])
        clean_medians.append(np.median(c_finals) if c_finals else 1e-16)
        
        # Noisy terminal errors across all solvers
        n_key = (dim, noisy_std, p_id)
        n_finals = []
        if n_key in all_benchmark_data:
            for s, runs in all_benchmark_data[n_key].items():
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        n_finals.append(raw_vals[-1])
        noisy_medians.append(np.median(n_finals) if n_finals else 1e-16)

    fig1 = go.Figure()
    
    # Clean bar (Solid Blue)
    fig1.add_trace(go.Bar(
        name="<b>Clean Evaluation (σ=0.0)</b>",
        x=problem_labels,
        y=np.maximum(clean_medians, 1e-16),
        marker=dict(
            color="#2B5C8F",
            line=dict(color="#1B3A5B", width=1.5)
        )
    ))
    
    # Noisy bar (Hatched Orange)
    fig1.add_trace(go.Bar(
        name="<b>Noisy Evaluation (σ=0.05)</b>",
        x=problem_labels,
        y=np.maximum(noisy_medians, 1e-16),
        marker=dict(
            color="#D95F02",
            pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8),
            line=dict(color="#8C3800", width=1.5)
        )
    ))

    fig1.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 1: Benchmark Problem Difficulty Under Stochastic Noise Extension — {dim}D</b><br><sup>Median Terminal Optimization Precision (Δy) Across Solvers by Landscape Class</sup>",
            x=0.02,
            y=0.96,
            font=dict(size=14, color="#2c3e50", family="Inter, Helvetica, Arial, sans-serif")
        ),
        xaxis=dict(
            title="<b>BBOB Landscape Class</b>",
            tickfont=dict(size=11)
        ),
        yaxis=dict(
            type="log",
            title="<b>Median Final Error log₁₀(Δy)</b>",
            range=[-16, 4],
            showgrid=True,
            gridwidth=1,
            gridcolor="#EAEAEA"
        ),
        barmode="group",
        bargap=0.25,
        bargroupgap=0.1,
        width=950,
        height=540,
        margin=dict(l=65, r=30, t=95, b=65),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=12, color="#333333"),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1.0,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="rgba(0,0,0,0.15)",
            borderwidth=1
        )
    )

    out_p = comparative_dir(dim) / 'figure_1_benchmark_validation.png'
    fig1.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 1 generated for all dimensions in results/figures/comparative/{dim}D/.')


✅ Thesis Figure 1 generated for all dimensions in results/figures/comparative/{dim}D/.


In [7]:
# ── THESIS Figure 2: Empirical Convergence Trajectories & ECDFs (RQ2 & RQ3) ──
eval_grid = np.logspace(0, 5, 200)
targets = np.logspace(-8, 2, 100)
grid_coords = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]

def render_figure_2_trajectories(dim: int, noise_std: float, label_env: str):
    subplot_titles = [
        f"<b>{BBOB_NAMES.get(p, f'f{p}')}</b><br><sup>{BBOB_CLASSES.get(p, '')}</sup>"
        for p in PROBLEM_IDS
    ] + ["<b>Overall Benchmark Portfolio</b><br><sup>Empirical Target Hit Rate (ECDF)</sup>"]

    fig_comp = make_subplots(
        rows=2, cols=3,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.22
    )

    for idx, p_id in enumerate(PROBLEM_IDS):
        r_idx, c_idx = grid_coords[idx]
        key = (dim, noise_std, p_id)
        if key not in all_benchmark_data: continue
        solvers_data = all_benchmark_data[key]
        
        for s in ALL_SOLVERS_ORDER:
            if s not in solvers_data or not solvers_data[s]: continue
            runs = solvers_data[s]
            interpolated = [np.interp(eval_grid, evals, raw_vals, left=raw_vals[0], right=raw_vals[-1]) for evals, raw_vals in runs if len(evals) > 0]
            if not interpolated: continue
            
            arr = np.array(interpolated)
            med = np.median(arr, axis=0)
            q25 = np.percentile(arr, 25, axis=0)
            q75 = np.percentile(arr, 75, axis=0)
            
            col = SOLVER_COLORS.get(s, "#7f7f7f")
            hex_c = col.lstrip("#")
            rgb = tuple(int(hex_c[i:i+2], 16) for i in (0, 2, 4))
            rgba_fill = f"rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.15)"
            
            is_14b = "14B" in s
            is_7b = "7B" in s
            is_classical = s in SOLVERS_CLASSICAL
            show_leg = (idx == 0)
            
            # IQR shaded band
            fig_comp.add_trace(
                go.Scatter(
                    x=np.concatenate([eval_grid, eval_grid[::-1]]),
                    y=np.concatenate([np.maximum(q75, 1e-16), np.maximum(q25[::-1], 1e-16)]),
                    fill="toself",
                    fillcolor=rgba_fill,
                    line=dict(color="rgba(255,255,255,0)"),
                    hoverinfo="skip",
                    showlegend=False
                ),
                row=r_idx, col=c_idx
            )
            
            # Median line
            fig_comp.add_trace(
                go.Scatter(
                    x=eval_grid,
                    y=np.maximum(med, 1e-16),
                    mode="lines",
                    name=s,
                    legendgroup="14B" if is_14b else ("7B" if is_7b else "Classical"),
                    legendgrouptitle_text="<b>LLaMEA-14B</b>" if (is_14b and s == SOLVERS_14B[0]) else ("<b>LLaMEA-7B</b>" if (is_7b and s == SOLVERS_7B[0]) else ("<b>Classical Baselines</b>" if (is_classical and s == SOLVERS_CLASSICAL[0]) else None)),
                    line=dict(
                        color=col,
                        width=2.8 if is_14b else (2.0 if is_7b else 1.5),
                        dash="solid" if (is_14b or is_7b) else "dash"
                    ),
                    showlegend=show_leg
                ),
                row=r_idx, col=c_idx
            )
            
        fig_comp.add_hline(y=1e-8, line_dash="dot", line_color="rgba(0,0,0,0.3)", row=r_idx, col=c_idx)
        fig_comp.update_xaxes(type="log", title="<b>Evaluations</b>", row=r_idx, col=c_idx)
        fig_comp.update_yaxes(type="log", range=[-16, 5], title="<b>Precision log₁₀(Δy)</b>" if c_idx == 1 else None, row=r_idx, col=c_idx)

    # 6th Subplot: Overall Target Hit Rate (ECDF)
    for s in ALL_SOLVERS_ORDER:
        all_s_terminals = []
        for p_id in PROBLEM_IDS:
            key = (dim, noise_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                all_s_terminals.extend(finals)
        if not all_s_terminals: continue
        
        hit_rates = [np.mean(np.array(all_s_terminals) <= t) for t in targets]
        is_14b = "14B" in s
        is_7b = "7B" in s
        col = SOLVER_COLORS.get(s, "#7f7f7f")
        fig_comp.add_trace(
            go.Scatter(
                x=targets, y=hit_rates, mode="lines", name=s,
                line=dict(
                    color=col,
                    width=2.8 if is_14b else (2.0 if is_7b else 1.5),
                    dash="solid" if (is_14b or is_7b) else "dash"
                ),
                showlegend=False
            ),
            row=2, col=3
        )
    fig_comp.update_xaxes(type="log", title="<b>Target Precision (Δy)</b>", row=2, col=3)
    fig_comp.update_yaxes(title="<b>Portfolio Hit Rate</b>", range=[0, 1.05], row=2, col=3)

    fig_comp.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Empirical Convergence Trajectories & Portfolio Competency ({label_env}) — {dim}D</b>",
            x=0.02, y=0.985,
            font=dict(size=14, color="#2c3e50", family="Inter, Helvetica, Arial, sans-serif")
        ),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=11, color="#333333"),
        margin=dict(l=65, r=30, t=250, b=55),
        width=1220, height=860,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.10, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.92)", bordercolor="rgba(0,0,0,0.12)", borderwidth=1,
            font=dict(size=10.5), groupclick="toggleitem"
        )
    )
    fig_comp.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False)
    fig_comp.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False)

    out_dir = model_std_dir(model_name, dim, noise_std)
    out_p = out_dir / "convergence_trajectories.png"
    fig_comp.write_image(str(out_p), scale=3)

def render_figure_2_ecdf(dim: int, noise_std: float, label_env: str):
    subplot_titles = [
        f"<b>{BBOB_NAMES.get(p, f'f{p}')}</b><br><sup>{BBOB_CLASSES.get(p, '')}</sup>"
        for p in PROBLEM_IDS
    ] + ["<b>Overall Benchmark Portfolio</b><br><sup>Empirical Target Hit Rate (All Problems)</sup>"]

    fig_ecdf = make_subplots(
        rows=2, cols=3,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.22
    )

    for idx, p_id in enumerate(PROBLEM_IDS):
        r_idx, c_idx = grid_coords[idx]
        key = (dim, noise_std, p_id)
        if key not in all_benchmark_data: continue
        solvers_data = all_benchmark_data[key]
        
        for s in ALL_SOLVERS_ORDER:
            if s not in solvers_data or not solvers_data[s]: continue
            runs = solvers_data[s]
            finals = [r[1][-1] for r in runs if len(r[1]) > 0]
            if not finals: continue
            
            hit_rates = [np.mean(np.array(finals) <= t) for t in targets]
            col = SOLVER_COLORS.get(s, "#7f7f7f")
            is_14b = "14B" in s
            is_7b = "7B" in s
            is_classical = s in SOLVERS_CLASSICAL
            show_leg = (idx == 0)
            
            fig_ecdf.add_trace(
                go.Scatter(
                    x=targets, y=hit_rates, mode="lines", name=s,
                    legendgroup="14B" if is_14b else ("7B" if is_7b else "Classical"),
                    legendgrouptitle_text="<b>LLaMEA-14B</b>" if (is_14b and s == SOLVERS_14B[0]) else ("<b>LLaMEA-7B</b>" if (is_7b and s == SOLVERS_7B[0]) else ("<b>Classical Baselines</b>" if (is_classical and s == SOLVERS_CLASSICAL[0]) else None)),
                    line=dict(
                        color=col,
                        width=2.8 if is_14b else (2.0 if is_7b else 1.5),
                        dash="solid" if (is_14b or is_7b) else "dash"
                    ),
                    showlegend=show_leg
                ),
                row=r_idx, col=c_idx
            )
        fig_ecdf.update_xaxes(type="log", title="<b>Target Precision (Δy)</b>", row=r_idx, col=c_idx)
        fig_ecdf.update_yaxes(title="<b>Hit Rate (ECDF)</b>" if c_idx == 1 else None, range=[0, 1.05], row=r_idx, col=c_idx)

    # 6th Subplot: Overall Portfolio ECDF
    for s in ALL_SOLVERS_ORDER:
        all_s_terminals = []
        for p_id in PROBLEM_IDS:
            key = (dim, noise_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                all_s_terminals.extend(finals)
        if not all_s_terminals: continue
        
        hit_rates = [np.mean(np.array(all_s_terminals) <= t) for t in targets]
        is_14b = "14B" in s
        is_7b = "7B" in s
        col = SOLVER_COLORS.get(s, "#7f7f7f")
        fig_ecdf.add_trace(
            go.Scatter(
                x=targets, y=hit_rates, mode="lines", name=s,
                line=dict(
                    color=col,
                    width=2.8 if is_14b else (2.0 if is_7b else 1.5),
                    dash="solid" if (is_14b or is_7b) else "dash"
                ),
                showlegend=False
            ),
            row=2, col=3
        )
    fig_ecdf.update_xaxes(type="log", title="<b>Target Precision (Δy)</b>", row=2, col=3)
    fig_ecdf.update_yaxes(title="<b>Portfolio Hit Rate</b>", range=[0, 1.05], row=2, col=3)

    fig_ecdf.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Empirical Target Precision Hit Rates (ECDF) ({label_env}) — {dim}D</b>",
            x=0.02, y=0.985,
            font=dict(size=14, color="#2c3e50", family="Inter, Helvetica, Arial, sans-serif")
        ),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=11, color="#333333"),
        margin=dict(l=65, r=30, t=250, b=55),
        width=1220, height=860,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.10, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.92)", bordercolor="rgba(0,0,0,0.12)", borderwidth=1,
            font=dict(size=10.5), groupclick="toggleitem"
        )
    )
    fig_ecdf.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False)
    fig_ecdf.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False)

    out_dir = model_std_dir(model_name, dim, noise_std)
    out_p = out_dir / "target_precision_ecdf.png"
    fig_ecdf.write_image(str(out_p), scale=3)

for model_name in DISCOVERED_MODELS:
    for dim in all_dims:
        render_figure_2_trajectories(dim, 0.0, 'Clean σ=0.0')
        render_figure_2_trajectories(dim, 0.05, 'Noisy σ=0.05')
        render_figure_2_ecdf(dim, 0.0, 'Clean σ=0.0')
        render_figure_2_ecdf(dim, 0.05, 'Noisy σ=0.05')

print('✅ Empirical Trajectories and ECDFs generated in results/figures/Qwen_14B/{dim}D/std_X/ for all dimensions.')


✅ Empirical Trajectories and ECDFs generated in results/figures/Qwen_14B/{dim}D/std_X/ for all dimensions.


In [8]:
# ── THESIS Figure 3: Cross-Environment Noise Robustness Profile (RQ3 Hero) ────
clean_std = 0.0
noisy_stds = [s for s in all_noise_stds if s > 0]
noisy_std = noisy_stds[0] if noisy_stds else 0.05

for dim in all_dims:
    fig_rob = go.Figure()
    
    solvers_in_plot = [s for s in ALL_SOLVERS_ORDER]
    clean_rates = []
    noisy_rates = []
    valid_solvers = []
    
    for s in solvers_in_plot:
        # Calculate clean success rate
        c_succ, c_tot = 0, 0
        for p_id in PROBLEM_IDS:
            key = (dim, clean_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        c_tot += 1
                        if raw_vals[-1] <= 1e-8:
                            c_succ += 1
        
        # Calculate noisy success rate
        n_succ, n_tot = 0, 0
        for p_id in PROBLEM_IDS:
            key = (dim, noisy_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        n_tot += 1
                        if raw_vals[-1] <= 1e-8:
                            n_succ += 1
                            
        if c_tot > 0 and n_tot > 0:
            clean_rates.append(c_succ / c_tot * 100.0)
            noisy_rates.append(n_succ / n_tot * 100.0)
            valid_solvers.append(s)

    if not valid_solvers: continue

    y_pos = np.arange(len(valid_solvers))
    
    # Add vertical reference drops (connectors)
    for idx, s in enumerate(valid_solvers):
        c_val = clean_rates[idx]
        n_val = noisy_rates[idx]
        drop = c_val - n_val
        line_color = "#2CA02C" if drop <= 10 else ("#D62728" if drop >= 25 else "#7F7F7F")
        
        fig_rob.add_trace(go.Scatter(
            x=[n_val, c_val],
            y=[s, s],
            mode="lines",
            line=dict(color=line_color, width=3.5),
            hoverinfo="skip",
            showlegend=False
        ))

    # Add Clean markers (Open Circle)
    fig_rob.add_trace(go.Scatter(
        x=clean_rates,
        y=valid_solvers,
        mode="markers",
        name="<b>Clean Success Rate (σ=0.0)</b>",
        marker=dict(
            symbol="circle-open",
            size=14,
            color="#2B5C8F",
            line=dict(width=2.5, color="#2B5C8F")
        )
    ))

    # Add Noisy markers (Filled Circle)
    fig_rob.add_trace(go.Scatter(
        x=noisy_rates,
        y=valid_solvers,
        mode="markers",
        name="<b>Noisy Success Rate (σ=0.05)</b>",
        marker=dict(
            symbol="circle",
            size=14,
            color="#D95F02",
            line=dict(width=1.5, color="#8C3800")
        )
    ))

    fig_rob.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 3: Cross-Environment Algorithm Robustness & Degradation Profile — {dim}D</b><br><sup>Paired Comparison of Precision Success Rate (Δy ≤ 10⁻⁸) on Clean vs. Noisy Landscapes</sup>",
            x=0.02,
            y=0.96,
            font=dict(size=14, color="#2c3e50", family="Inter, Helvetica, Arial, sans-serif")
        ),
        xaxis=dict(
            title="<b>Success Rate (%)</b>",
            range=[-2, 105],
            showgrid=True,
            gridwidth=1,
            gridcolor="#EAEAEA"
        ),
        yaxis=dict(
            title="<b>Optimization Solver</b>",
            autorange="reversed",
            tickfont=dict(size=11)
        ),
        width=980,
        height=520,
        margin=dict(l=190, r=40, t=95, b=65),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=12, color="#333333"),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1.0,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="rgba(0,0,0,0.15)",
            borderwidth=1
        )
    )

    out_p = comparative_dir(dim) / 'figure_3_robustness.png'
    fig_rob.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 3 generated for all dimensions in results/figures/comparative/{dim}D/.')


✅ Thesis Figure 3 generated for all dimensions in results/figures/comparative/{dim}D/.


In [9]:
# ── THESIS Figure 4: Prompt Scaffolding Ablation on LLaMEA-14B (RQ2/3 Ablation) 
clean_std = 0.0
noisy_stds = [s for s in all_noise_stds if s > 0]
noisy_std = noisy_stds[0] if noisy_stds else 0.05

for dim in all_dims:
    strategies = ["baseline", "guided", "thinking", "vectorization"]
    strat_labels = ["<b>Baseline</b>", "<b>Guided</b>", "<b>Thinking</b>", "<b>Vectorization</b>"]
    
    clean_succ_rates = []
    noisy_succ_rates = []
    
    for strat in strategies:
        s_name = f"LLaMEA-14B / {strat}"
        
        # Clean
        c_succ, c_tot = 0, 0
        for p_id in PROBLEM_IDS:
            key = (dim, clean_std, p_id)
            if key in all_benchmark_data and s_name in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s_name]
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        c_tot += 1
                        if raw_vals[-1] <= 1e-8:
                            c_succ += 1
        clean_succ_rates.append(c_succ / c_tot * 100.0 if c_tot > 0 else 0.0)
        
        # Noisy
        n_succ, n_tot = 0, 0
        for p_id in PROBLEM_IDS:
            key = (dim, noisy_std, p_id)
            if key in all_benchmark_data and s_name in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s_name]
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        n_tot += 1
                        if raw_vals[-1] <= 1e-8:
                            n_succ += 1
        noisy_succ_rates.append(n_succ / n_tot * 100.0 if n_tot > 0 else 0.0)

    fig4 = go.Figure()
    
    # Clean bar
    fig4.add_trace(go.Bar(
        name="<b>Clean (σ=0.0)</b>",
        x=strat_labels,
        y=clean_succ_rates,
        marker=dict(
            color="#2B5C8F",
            line=dict(color="#1B3A5B", width=1.5)
        )
    ))
    
    # Noisy bar
    fig4.add_trace(go.Bar(
        name="<b>Noisy (σ=0.05)</b>",
        x=strat_labels,
        y=noisy_succ_rates,
        marker=dict(
            color="#D95F02",
            pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8),
            line=dict(color="#8C3800", width=1.5)
        )
    ))

    fig4.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 4: Impact of Prompt Scaffolding on LLaMEA-14B Synthesis Quality — {dim}D</b><br><sup>Precision Success Rate (Δy ≤ 10⁻⁸) Across Clean vs. Stochastic Benchmark Regimes</sup>",
            x=0.02,
            y=0.96,
            font=dict(size=14, color="#2c3e50", family="Inter, Helvetica, Arial, sans-serif")
        ),
        xaxis=dict(
            title="<b>Prompt Scaffolding Strategy</b>",
            tickfont=dict(size=11)
        ),
        yaxis=dict(
            title="<b>Benchmark Success Rate (%)</b>",
            range=[0, 105],
            showgrid=True,
            gridwidth=1,
            gridcolor="#EAEAEA"
        ),
        barmode="group",
        bargap=0.28,
        bargroupgap=0.1,
        width=880,
        height=500,
        margin=dict(l=65, r=30, t=95, b=65),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=12, color="#333333"),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1.0,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="rgba(0,0,0,0.15)",
            borderwidth=1
        )
    )

    out_p = model_fig_dir(next((m for m in DISCOVERED_MODELS if "14b" in m.lower()), DISCOVERED_MODELS[0]), dim) / 'figure_4_scaffolding.png'
    fig4.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 4 generated for all dimensions in results/figures/Qwen_14B/{dim}D/.')


✅ Thesis Figure 4 generated for all dimensions in results/figures/Qwen_14B/{dim}D/.
